# Calidad de Datos y Backfill

Objetivos:
- Aplicar validaciones básicas (nulos, duplicados, rangos).
- Patrón de aserciones reutilizable.
- Estrategias de backfill (reprocesos históricos) y consideraciones de costo.

## 1. Validaciones Fundamentales

Tipos comunes:
- Not Null / Completeness
- Uniqueness
- Range / Domain
- Consistency Cross-column
- Freshness (se cubre antes con SLOs)

In [ ]:
import pandas as pd
data = pd.DataFrame({
  'fecha': ['2025-01-01','2025-01-01','2025-01-02', None],
  'tienda_id': [10,10,11,12],
  'importe': [150.0,150.0,3000.0,200.0],
  'unidades': [3,3,60,4]
})
print(data)

## 2. Aserciones Reutilizables

In [ ]:
def assert_not_null(df, cols):
    for c in cols:
        if df[c].isna().any():
            raise AssertionError(f'Null values found in {c}')
def assert_unique(df, cols):
    if df.duplicated(subset=cols).any():
        raise AssertionError(f'Duplicates found for key {cols}')
def assert_range(df, col, min_v, max_v):
    if ((df[col] < min_v) | (df[col] > max_v)).any():
        raise AssertionError(f'Out-of-range values in {col}')

try:
    assert_not_null(data, ['fecha','tienda_id'])
except AssertionError as e:
    print('Error calidad:', e)

try:
    assert_unique(data, ['fecha','tienda_id'])
except AssertionError as e:
    print('Error calidad:', e)

try:
    assert_range(data, 'importe', 0, 100000)
    assert_range(data, 'unidades', 0, 10000)
except AssertionError as e:
    print('Error calidad:', e)

## 3. Reporte de Resultados

In [ ]:
def run_quality_suite(df):
    results = []
    tests = [
        ('not_null(fecha,tienda_id)', lambda d: not d['fecha'].isna().any() and not d['tienda_id'].isna().any()),
        ('unique(fecha,tienda_id)', lambda d: not d.duplicated(subset=['fecha','tienda_id']).any()),
        ('range(importe)', lambda d: ((d['importe']>=0) & (d['importe']<=100000)).all()),
        ('range(unidades)', lambda d: ((d['unidades']>=0) & (d['unidades']<=10000)).all()),
    ]
    for name, fn in tests:
        ok = fn(df)
        results.append({'test': name, 'status': 'PASS' if ok else 'FAIL'})
    return pd.DataFrame(results)

print(run_quality_suite(data))

## 4. Estrategias de Backfill

Backfill = reprocesar histórico para corregir calidad, recalcular métricas o incorporar lógica nueva.

Patrones:
- Reproceso completo (costoso)
- Reproceso incremental por ventanas (e.g. 7 días)
- Recalculación bajo demanda (on-demand backfill)
- Shadow pipeline + switch (ejecutar nuevo pipeline en paralelo)

Consideraciones de costo:
- Almacenamiento temporal adicional
- Compute (Spark/Flink/DB MPP) horas/picos
- Impacto en SLAs si compite con cargas productivas

Buenas prácticas:
- Etiquetar datasets por versión
- Registrar lineage del reproceso
- Validar calidad antes de sustituir dataset oficial

## 5. Ejercicio Rápido

Diseña un plan de backfill para corregir importe_total de los últimos 30 días sin afectar SLAs. Pasos sugeridos:
1. Identificar ventanas y particiones afectadas
2. Ejecutar cálculo incremental en entorno aislado
3. Validar calidad con suite previa
4. Registrar lineage y versión
5. Reemplazar particiones en Gold

In [ ]:
plantilla = '''
# Plan Backfill (borrador)
Ventana: [fechas]
Motivo: [corrección / nueva métrica]
Estrategia: [incremental / completo]
Validaciones: [tests]
Riesgos: [costo, SLA]
Mitigación: [ventanas nocturnas, throttling]
'''
print(plantilla)

## 6. Resumen

- Validaciones simples previenen errores comunes.
- Aserciones reutilizables crean consistencia.
- Backfill requiere planeación para evitar impacto en producción.
- Lineage/versionado permiten trazabilidad de reprocesos.

---
Referencias:
- Great Expectations (conceptos)
- OpenLineage
- Apache Airflow best practices